In [1]:
import numpy as np
import pandas as pd
import h5py

from Account import *
from Agent import *
# from DataAsset import * 
from Exchnage import *
from Updater import *

In [7]:
import numpy as np
import h5py
import pickle
import dask.array as da


def make_data(filename, name, data_df, fields=None, dtype=None, dates=None):
    """
    sql에서 읽은 dataframe을 (날짜, 종목코드, 필드)로 구성된 3-dimensional array로 변환한다.
    :param data_df: pandas.dataframe, 종목코드와 날짜를 각각 table과 index로 갖는 pandas dataframe
    :param dates:
    :return: numpy.array, (list, list, list), 두번째 tuple은 각각 날짜, 종목코드, 필드의 리스트로 구성된다.
    """

    if dates is None:
        dates = data_df["A005930"].index  # A005930 : 삼성전자
    else:
        dates = dates

    codes = list(data_df.keys())
    list_data = list()

    fields = list(data_df[codes[0]].columns)
    if dtype == "stock":
        fields = ["현재가", "시가", "고가", "저가", "대비", "거래량(주)", "거래대금(원)", "상장시가총액(원)", "시장구분"]

    for code in codes:
        dummy = data_df[code][fields].reindex(dates).fillna(np.nan)
        list_data.append(np.array(dummy).reshape(len(dates), 1, -1))

    dates = list(dates)
    array = np.concatenate(list_data, axis=1)

    if dtype == "stock":

        idx = fields.index("시장구분")

        array[:, :, idx][np.where(array[:, :, idx] == "코스피")] = 0
        array[:, :, idx][np.where(array[:, :, idx] == "코스닥")] = 1
        array = array.astype('f')

    f = h5py.File(filename, "a")
    try:
        f.create_dataset(name, data=array)
    except:
        del f[name]
        f.close()

        f = h5py.File(filename, "a")
        f.create_dataset(name, data=array)
    f.close()

    with open("%s-%s.axis" % (filename, name), "wb") as f:
        pickle.dump((dates, codes, fields), f)


def load_data(file, name, chunks=5, in_memory=False):
    with open("%s-%s.axis" % (file.filename, name), "rb") as f:
        axis = pickle.load(f)

    array = file[name]
    if in_memory:
        array = array[:]

    array = da.from_array(array, chunks=(chunks, len(axis[1]), len(axis[2])))

    return array, axis

class DataAsset:
    def __init__(self, array, axis, chunks=300):
        #         array, axis = make_data(data_df)

        self.dates, self.codes, self.fields = list(axis[0]), list(axis[1]), list(axis[2])
        self.array = array

        self._chunks = chunks
        self._dates_chunk = []

        self._date = None

    def get_info(self, date, num=1, codes=None, fields=None):
        """

        :param date: Pandas.Timestamp
        :param num: int, 반환할 과거 일수
        :param codes: list, 반환할 종목코드들의 리스트
        :param fields: list, 반환할 필드들의 리스트
        :return: numpy.array
        """
        if date not in self._dates_chunk:
            self._make_chunk(date)
        idx_date = self._dates_chunk.index(date)
        array = self._array_chunk[max(0, idx_date - num + 1):idx_date + 1]

#         idx_date = self.dates.index(date)
#         array = self.array[max(0, idx_date - num + 1):idx_date + 1]

        if codes is not None:
            idx_codes = [self.codes.index(code) for code in codes]
            array = array[:, idx_codes, :]
        if fields is not None:
            idx_fields = [self.fields.index(field) for field in fields]
            array = array[:, :, idx_fields]

        if num == 1:
            array = array[0]

        return array

    def _make_chunk(self):
        idx_date = self.dates.index(self._date)
        self._dates_chunk = self.dates[max(0, idx_date - self._chunks):idx_date + self._chunks]
        self._array_chunk = self.array[max(0, idx_date - self._chunks):idx_date + self._chunks].compute()

    def update_date(self, date):
        self._date = date
        if date not in self._dates_chunk:
            self._make_chunk()
            
#     def update_date(self, date):
#         pass

    def init(self, date):
        self.update_date(date)


# with pandas dataframe

In [8]:
%%time
f = h5py.File("./data/stock_info.hdf5", "r")
array_stock, axis_stock = load_data(f, "stock", chunks=10, in_memory=False)
array_value, axis_value = load_data(f, "value", chunks=10, in_memory=False)

data_stock = DataAsset(array_stock, axis_stock, chunks=600)
data_value = DataAsset(array_value, axis_value, chunks=600)

updater = Updater(pd.Timestamp(2003, 1, 1), data_stock.dates)

# 거래소 생성
exchange_stock = Exchange()
exchange_stock.set_DataAsset(data_stock)

# 주식 계좌 생성
stock_account = StockAccount(exchange_stock, 출력=False)

# 거래 에이전트 생성 및 주식 계좌 등록
agent = Agent(1e8, 출력=False)
agent.set_account("stock", stock_account)

# 날짜가 변할시 업데이트 요청
updater.set_data(data_stock)
updater.set_data(data_value)

updater.set_agent(agent)
updater.set_exchange(exchange_stock)

updater.initialization()
columns = ["상장시가총액(원)", "지배주주순이익(원)(직전4분기)", "지배주주지분(원)",
           "현금흐름(원)(직전4분기)", "매출액(원)(직전4분기)"]

while updater._date != updater._list_date[-1]:
    fin_stat = data_value.get_info(updater._date, num=2,
                                   fields=columns)
    fin_stat = data_value.get_info(updater._date, num=2,
                               fields=columns)
    fin_stat = data_value.get_info(updater._date, num=2,
                                   fields=columns)

    updater.update()

Wall time: 17.6 s


In [15]:
a = list(np.arange(1000))
b = np.arange(1000)

In [31]:
%%timeit
a.index(3)

482 ns ± 25.9 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


In [25]:
%%timeit
np.where(b == 3)[0][0]

2.17 µs ± 204 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


In [24]:
np.where(b.reshape(1, -1) == 3)[0]

array([0], dtype=int64)

In [30]:
b == 0  1

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [2]:
import cython

In [10]:
%load_ext Cython

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


In [17]:
def cal2():
    b = list(np.arange(3000))
    a = [b.index(x) for x in np.arange(1000)]

In [28]:
%%cython
import numpy as np


cdef cal():
    a = [b.index(x) for x in np.arange(1000)]

In [29]:
%%timeit
cal()

7 ms ± 271 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [18]:
%%timeit
cal2()

7.74 ms ± 680 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
